In [1]:
from database.adatabase import ADatabase
import pandas as pd
from modeler.modeler import Modeler as m
from CFA.cfa import CFA as cfa
import matplotlib.pyplot as plt
from processor.processor import Processor as processor
from xgboost.sklearn import XGBRegressor
from tqdm import tqdm
import warnings
warnings.simplefilter(action="ignore")
import pickle
from datetime import datetime, timedelta, timezone

In [2]:
db = ADatabase("financial")
market = ADatabase("market")
fed = ADatabase("fed")
market.connect()
sp500 = market.retrieve("sp500")
market.disconnect()

In [3]:
holding_period = 5
tickers = sp500["ticker"].values
factors = ["assets","liabilities","stockholdersequity","adjclose"]
positions = len(sp500["GICS Sector"].unique())
training_year = 2019
training_years = 7
hedge_percentage = 0.05

In [4]:
market.connect()
model = XGBRegressor(booster="gbtree",objective ='reg:squarederror', colsample_bytree = 0.3, learning_rate = 0.1,max_depth = 5, alpha = 10, n_estimators = 100,  verbosity=0,refit=False)
for ticker in tqdm(tickers,desc="model_prep"):
    try:
        cik = sp500[sp500["ticker"]==ticker]["CIK"].item()
        ticker_prices = processor.column_date_processing(market.query("prices",{"ticker":ticker}))
        filings = market.query("filings",{"cik":cik})
        ticker_prices.sort_values("date",inplace=True)
        ticker_prices = processor.merge(ticker_prices,filings,["year","quarter"]).ffill()
        ticker_prices = ticker_prices.drop(["date","adsh"],axis=1).groupby(["year","quarter","ticker"]).mean().reset_index()
        ticker_prices.sort_values(["year","quarter"],inplace=True)
        ticker_prices["y"] = ticker_prices["adjclose"].shift(-1)
        model_data = ticker_prices[(ticker_prices["year"]<=training_year) & (ticker_prices["year"]>=training_year-training_years)].dropna().reset_index(drop=True)
        model.fit(model_data[factors],model_data["y"])
    except Exception as e:
        print(str(e))
        continue
market.disconnect()

model_prep:   1%|          | 3/503 [00:00<01:01,  8.14it/s]

'year'
'year'
'year'
'year'


model_prep:   1%|▏         | 7/503 [00:00<00:41, 11.99it/s]

'year'
'year'
'year'


model_prep:   2%|▏         | 9/503 [00:00<00:47, 10.48it/s]

...

model_prep:  99%|█████████▉| 497/503 [00:35<00:00, 19.25it/s]

'year'
'year'
'year'
'year'
'year'


model_prep: 100%|█████████▉| 501/503 [00:35<00:00, 19.40it/s]

'year'
'year'
'year'
'year'
'year'


model_prep: 100%|██████████| 503/503 [00:35<00:00, 14.08it/s]


In [5]:
market.connect()
prices = []
for ticker in tqdm(tickers,desc="model_prep"):
    try:
        cik = sp500[sp500["ticker"]==ticker]["CIK"].item()
        ticker_prices = processor.column_date_processing(market.query("prices",{"ticker":ticker}))
        filings = market.query("filings",{"cik":cik})
        filings = filings.sort_values(["year","quarter"]).ffill()
        ticker_prices.sort_values("date",inplace=True)
        predictions = processor.merge(ticker_prices,filings,["year","quarter"]).ffill()
        predictions = predictions.drop(["date","adsh"],axis=1).groupby(["year","quarter","ticker"]).mean().reset_index()
        predictions.sort_values(["year","quarter"],inplace=True)
        predictions["prediction"] = model.predict(predictions[factors])
        predictions["quarter"] = [x+1 if x < 4 else 1 for x in predictions["quarter"]]
        predictions["year"] = [row[1]["year"] if row[1]["quarter"] < 4 else row[1]["year"] + 1 for row in predictions.iterrows()]
        simulation = ticker_prices[(ticker_prices["year"]>training_year)].reset_index(drop=True)
        simulation = simulation.merge(predictions[["year","quarter","ticker","prediction"]],on=["year","quarter","ticker"])
        simulation.sort_values("date",inplace=True)
        simulation["expected_return"] = (simulation["prediction"] - simulation["adjclose"]) / simulation["adjclose"]
        simulation["buy_price"] = simulation["adjclose"].shift(-1)
        simulation["buy_date"] = simulation["date"].shift(-1)
        simulation["sell_price"] = simulation["adjclose"].shift(-holding_period)
        simulation["sell_date"] = simulation["date"].shift(-holding_period)
        simulation["return"] = (simulation["sell_price"] - simulation["buy_price"]) / simulation ["buy_price"] * (1/positions)
        simulation["return"] = [max(float(-hedge_percentage/positions),x) for x in simulation["return"]]
        prices.append(simulation)
    except Exception as e:
        print(str(e))
        continue
market.disconnect()

model_prep:   1%|          | 4/503 [00:00<00:32, 15.59it/s]

'year'
'year'
'year'
'year'


model_prep:   2%|▏         | 8/503 [00:00<00:28, 17.35it/s]

'year'
'year'
'year'
'year'
'year'


model_prep:   3%|▎         | 13/503 [00:00<00:25, 18.98it/s]

...

'year'
'year'
'year'
'year'


model_prep:  99%|█████████▉| 499/503 [00:32<00:00, 15.49it/s]

'year'
'year'
'year'
'year'


model_prep: 100%|██████████| 503/503 [00:33<00:00, 15.20it/s]

'year'
'year'


In [6]:
sim = pd.concat(prices).reset_index(drop=True)
sim.sort_values("date",inplace=True)
sim = processor.merge(sim,sp500,on="ticker")
sim = cfa.cfa(sim,holding_period)

ValueError: No objects to concatenate

In [ ]:
## backtest
trades = sim[sim["weekday"]==4]
trades = trades[trades["week"] % int(holding_period/5) == 0]
trades = trades.sort_values("excess_return",ascending=False).groupby(["date","GICS Sector"]).first().reset_index()

In [ ]:
trades = processor.column_date_processing(trades)

In [ ]:
portfolio = trades[["date","return"]].groupby("date").sum().reset_index()
portfolio.sort_values("date",inplace=True)
portfolio = portfolio[portfolio["date"]<portfolio["date"].max()]
portfolio["return"] = portfolio["return"] + 1
portfolio["cr"] = portfolio["return"].cumprod()

In [ ]:
fed.connect()
bench = fed.retrieve("sp500")
fed.disconnect()
bench["date"] = pd.to_datetime(bench["date"],utc=True)
bench["value"] = [float(x) for x in bench["value"]]
portfolio = processor.column_date_processing(portfolio)
portfolio = processor.merge(portfolio,bench,on="date")
portfolio.dropna(inplace=True)
portfolio["bcr"] = (portfolio["value"] - portfolio["value"].iloc[0]) / portfolio["value"].iloc[0] + 1
portfolio["beta"] = portfolio["cr"].cov(portfolio["value"])
portfolio["treynor"] = portfolio["cr"] / portfolio["beta"]

In [ ]:
portfolio["return"].min()

In [ ]:
plt.plot(portfolio["date"].values,portfolio["cr"].values)
plt.plot(portfolio["date"].values,portfolio["bcr"].values)
plt.show()

In [ ]:
recommendations = trades.sort_values("date").tail(positions)

In [ ]:
recommendations

In [ ]:
db.connect()
db.drop('portfolios')
db.drop('trades')
db.drop('recommendations')
db.store("portfolio",portfolio)
db.store("trades",trades)
db.store("recommendations",recommendations)
db.disconnect()